# RoboReplan — GRPO Training
**OpenEnv Hackathon submission**

Trains Qwen2.5-0.5B to plan, replan, and recover in a tabletop robot manipulation environment.

Run on: Northflank H100 or Google Colab (T4 for small model)

In [ ]:
# Install dependencies
!pip install -q trl transformers torch datasets openenv-core pydantic numpy
!git clone https://huggingface.co/spaces/YOUR_HF_USERNAME/robo-replan
%cd robo-replan

In [ ]:
# Verify environment works
import sys
sys.path.insert(0, '.')

from server.config import EnvConfig
from server.environment import TabletopPlanningEnv

env = TabletopPlanningEnv(config=EnvConfig.easy())
obs = env.reset()
print('Environment OK')
print('Instruction:', obs.instruction)
print('Objects:', [(o.name, 'reachable' if o.reachable else 'BLOCKED') for o in obs.visible_objects])
print('Valid actions:', obs.valid_actions)

In [ ]:
# ── Baseline: random policy (BEFORE training) ─────────────────────────
import random
from server.models import Action

ACTIONS = [a.value for a in Action]

def eval_policy(policy_fn, n_episodes=50):
    env = TabletopPlanningEnv(config=EnvConfig.easy())
    rewards, successes = [], []
    for _ in range(n_episodes):
        obs = env.reset()
        total = 0
        for _ in range(20):
            action = policy_fn(obs)
            r = env.step(action)
            total += r.reward
            obs = r.observation
            if r.done: break
        rewards.append(total)
        successes.append(env._all_goals_complete())
    return {
        'success_rate': sum(successes)/n_episodes,
        'avg_reward': sum(rewards)/n_episodes,
        'rewards': rewards
    }

def random_policy(obs):
    return random.choice(ACTIONS)

baseline = eval_policy(random_policy, n_episodes=50)
print(f'BEFORE TRAINING (random):')
print(f'  Success rate: {baseline["success_rate"]:.0%}')
print(f'  Avg reward:   {baseline["avg_reward"]:.2f}')

In [ ]:
# ── Build training dataset from oracle rollouts ───────────────────────
from datasets import Dataset

SYSTEM = (
    "You are a robot planning agent. Complete tabletop manipulation tasks "
    "by choosing ONE action per step.\n\n"
    "Actions: SCAN_SCENE | MOVE_TO_RED | MOVE_TO_BLUE | MOVE_TO_GREEN | "
    "MOVE_TO_YELLOW | MOVE_TO_PURPLE | PICK | PLACE_BIN_A | PLACE_BIN_B | CLEAR_BLOCKER\n\n"
    "Reply with ONLY the action name."
)

def obs_to_messages(obs):
    objects = ', '.join(
        f"{o.name}({'reachable' if o.reachable else 'BLOCKED'})"
        for o in obs.visible_objects
    )
    history = ' → '.join(obs.action_history[-5:]) or 'none'
    failures = '; '.join(obs.known_failures) or 'none'
    subgoals = '; '.join(obs.completed_subgoals) or 'none yet'
    valid = ', '.join(obs.valid_actions) if obs.valid_actions else 'any'
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': (
            f"Instruction: {obs.instruction}\n"
            f"Scene: {objects}\n"
            f"Holding: {obs.holding or 'nothing'}\n"
            f"Progress: {obs.goal_progress:.0%}  Remaining: {obs.goals_remaining}\n"
            f"Completed: {subgoals}\n"
            f"Failures: {failures}\n"
            f"History: {history}\n"
            f"Last: {obs.last_action or 'none'} → {obs.last_result or 'n/a'}\n"
            f"Valid now: {valid}\n"
            f"Steps left: {obs.steps_remaining}\n\n"
            f"Next action:"
        )}
    ]

cfg = EnvConfig.easy()
cfg.obs.include_oracle_hint = True

env = TabletopPlanningEnv(config=cfg)
rows = []
for _ in range(500):  # 500 oracle episodes
    obs = env.reset()
    for _ in range(20):
        if obs.oracle_hint:
            rows.append({
                'prompt': obs_to_messages(obs),
                'answer': obs.oracle_hint
            })
        action = obs.oracle_hint or 'SCAN_SCENE'
        r = env.step(action)
        obs = r.observation
        if r.done: break

dataset = Dataset.from_list(rows).train_test_split(test_size=0.1)
print(f'Dataset: {len(rows)} steps  ({len(dataset["train"])} train, {len(dataset["test"])} eval)')
print('Example prompt:', rows[0]['prompt'][1]['content'][:300])
print('Label:', rows[0]['answer'])

In [ ]:
# ── GRPO Training ─────────────────────────────────────────────────────
import torch
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading {MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16)

# ── Reward function: execute the model's chosen action in the env ─────
training_env = TabletopPlanningEnv(config=EnvConfig.easy())
_current_obs = [None]

def extract_action(text):
    text = text.strip().upper().replace('-', '_').replace(' ', '_')
    for a in ACTIONS:
        if a in text:
            return a
    return None

def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    for completion in completions:
        action = extract_action(completion if isinstance(completion, str)
                                else completion[0].get('content', ''))
        if action is None:
            rewards.append(-2.0)
            continue
        try:
            result = training_env.step(action)
            rewards.append(result.reward)
            if result.done:
                training_env.reset()
        except Exception:
            training_env.reset()
            rewards.append(-1.0)
    return rewards

training_env.reset()

grpo_config = GRPOConfig(
    output_dir='./outputs/grpo',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,
    max_new_tokens=12,
    logging_steps=5,
    save_steps=50,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=reward_fn,
    train_dataset=dataset['train'],
    processing_class=tokenizer,
)

print('Training...')
trainer.train()
trainer.save_model('./outputs/grpo_final')
print('Done → ./outputs/grpo_final')

In [ ]:
# ── AFTER training: evaluate trained model ────────────────────────────
from transformers import pipeline

pipe = pipeline('text-generation', model='./outputs/grpo_final',
                tokenizer=tokenizer, max_new_tokens=12, device_map='auto')

def trained_policy(obs):
    messages = obs_to_messages(obs)
    out = pipe(messages, return_full_text=False)[0]['generated_text']
    action = extract_action(out)
    return action or random.choice(ACTIONS)

after = eval_policy(trained_policy, n_episodes=50)
print(f'AFTER TRAINING:')
print(f'  Success rate: {after["success_rate"]:.0%}')
print(f'  Avg reward:   {after["avg_reward"]:.2f}')
print()
print(f'Improvement:')
print(f'  Success: {baseline["success_rate"]:.0%} → {after["success_rate"]:.0%}')
print(f'  Reward:  {baseline["avg_reward"]:.1f} → {after["avg_reward"]:.1f}')

In [ ]:
# ── Plot reward curve ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Before (random)', 'After (trained)'],
            [baseline['success_rate'], after['success_rate']],
            color=['#e05555', '#55c055'])
axes[0].set_ylabel('Success Rate')
axes[0].set_title('Task Completion')
axes[0].set_ylim(0, 1)
for i, v in enumerate([baseline['success_rate'], after['success_rate']]):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')

axes[1].bar(['Before (random)', 'After (trained)'],
            [baseline['avg_reward'], after['avg_reward']],
            color=['#e05555', '#55c055'])
axes[1].set_ylabel('Average Reward')
axes[1].set_title('Reward Improvement')
for i, v in enumerate([baseline['avg_reward'], after['avg_reward']]):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')

plt.suptitle('RoboReplan: GRPO Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_results.png')